# Разминка к ДЗ №1: кирпичики ML-пайплайна

Это **не** мини-версия ДЗ №1, а отработка отдельных «кирпичиков», из которых он собран —
каждый с автопроверкой. Освоив их здесь, в ДЗ №1 вы соберёте пайплайн на своих данных.

**Датасет фиксированный и офлайн** — `sklearn.load_breast_cancer` (бинарная классификация),
в него детерминированно добавлены категориальный признак и пропуски.

**Как работать.** Для каждого кирпичика три ячейки:
1. **задание** (markdown);
2. **реализация** — заполните тело функции вместо `# TODO`;
3. **проверка** — запустите: она напечатает `PASS/FAIL` по каждому критерию и сверит результат с эталоном.

Кирпичики: стратифицированный сплит · импутация без утечки · One-Hot · метрики руками ·
ROC-AUC при дисбалансе · кросс-валидация · GridSearchCV · важность признаков.

Сначала выполните две служебные ячейки ниже.

In [40]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, roc_auc_score)

SEED = 234

_data = load_breast_cancer(as_frame=True)
NUM_COLS = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
df = _data.frame[NUM_COLS + ["target"]].copy()
df["area_cat"] = pd.qcut(df["mean area"], q=3, labels=["small", "medium", "large"]).astype(object)
df.loc[np.arange(0, len(df), 13), "mean smoothness"] = np.nan
CAT_COLS = ["area_cat"]

y = df["target"]
X = df.drop(columns="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

Xn = X[NUM_COLS].copy()
Xn = Xn.fillna(Xn.median())

C_GRID = [0.01, 0.1, 1, 10]
PARAM_GRID = {"logisticregression__C": C_GRID}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
def make_estimator():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,target,area_cat
0,17.99,10.38,122.80,1001.0,NaN,0,large
1,20.57,17.77,132.90,1326.0,0.08474,0,large
2,19.69,21.25,130.00,1203.0,0.10960,0,large
3,11.42,20.38,77.58,386.1,0.14250,0,small
4,20.29,14.34,135.10,1297.0,0.10030,0,large


In [41]:
def run_checks(title, checks):
    """checks: список (описание, callable->bool). Печатает PASS/FAIL и падает, если не всё пройдено."""
    print(title)
    ok_count = 0
    for desc, cond in checks:
        try:
            ok = bool(cond())
        except Exception as e:
            ok, desc = False, f"{desc}  [ошибка: {type(e).__name__}: {e}]"
        print(f"  [{'PASS' if ok else 'FAIL'}] {desc}")
        ok_count += ok
    print(f"  -> {ok_count}/{len(checks)} проверок пройдено")
    assert ok_count == len(checks), "Не все проверки пройдены — доработайте функцию."

### Кирпичик 1 — стратифицированный train/test split

**`train_test_split(X, y, ...)`** — случайно делит выборку на обучающую и тестовую части.
`test_size` задаёт долю теста, `stratify=y` сохраняет доли классов в обеих частях,
`random_state` фиксирует разбиение (воспроизводимость).
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

**Задание.** Реализуйте `make_split(X, y)` с `test_size=0.25`, `stratify=y`, `random_state=SEED`.
Верните `X_train, X_test, y_train, y_test`. Сплит делается **до** любой предобработки.

In [3]:
def make_split(X, y):
    # TODO: верните стратифицированное разбиение (test_size=0.25, stratify=y, random_state=SEED)
    #       в порядке: X_train, X_test, y_train, y_test
    return train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

In [4]:
Xtr, Xte, ytr, yte = make_split(X, y)
_rXtr, _rXte, _rytr, _ryte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

run_checks("Кирпичик 1 — стратифицированный split", [
    ("Размер теста = 25%",                     lambda: len(Xte) == len(_rXte)),
    ("Train и test не пересекаются",           lambda: len(set(Xtr.index) & set(Xte.index)) == 0),
    ("Покрыты все объекты",                    lambda: len(Xtr) + len(Xte) == len(X)),
    ("Индексы теста = стратиф. эталон",        lambda: set(Xte.index) == set(_rXte.index)),
    ("Доля классов сохранена (стратификация)", lambda: abs(ytr.mean() - y.mean()) < 0.02),
])

Кирпичик 1 — стратифицированный split
  [PASS] Размер теста = 25%
  [PASS] Train и test не пересекаются
  [PASS] Покрыты все объекты
  [PASS] Индексы теста = стратиф. эталон
  [PASS] Доля классов сохранена (стратификация)
  -> 5/5 проверок пройдено


### Кирпичик 2 — заполнение пропусков без утечки

**`Series.median()`** — медиана столбца, пропуски (`NaN`) игнорируются.
[Документация →](https://pandas.pydata.org/docs/reference/api/pandas.Series.median.html)
**`Series.fillna(value)`** — заменяет `NaN` на переданное значение.
[Документация →](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html)
*(В реальном пайплайне то же делает [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) внутри `Pipeline`.)*

**Задание.** Реализуйте `impute_no_leak(train_col, test_col)`: посчитайте медиану **только по train**
и заполните ею пропуски и в train, и в test. Верните `(train_filled, test_filled)`.
Медиана по всем данным — это утечка.

In [5]:
def impute_no_leak(train_col: pd.Series, test_col: pd.Series):
    # TODO: посчитайте медиану ТОЛЬКО по train_col и заполните ею пропуски и в train, и в test
    train_col = train_col.fillna(train_col.median())
    test_col = test_col.fillna(train_col.median())
        
    return train_col, test_col

In [6]:
_tr, _te = X_train["mean smoothness"], X_test["mean smoothness"]
_trf, _tef = impute_no_leak(_tr, _te)
_train_med = _tr.median()
_full_med = pd.concat([_tr, _te]).median()

run_checks("Кирпичик 2 — импутация без утечки", [
    ("В train не осталось пропусков",              lambda: int(_trf.isna().sum()) == 0),
    ("В test не осталось пропусков",               lambda: int(_tef.isna().sum()) == 0),
    ("Пропуски заполнены медианой TRAIN",          lambda: np.allclose(_tef[_te.isna()].values, _train_med)),
    ("Медиана train != медианы всех данных (есть чем отличить утечку)",
        lambda: abs(_train_med - _full_med) > 1e-9),
    ("Не использована медиана всех данных (нет утечки)",
        lambda: not np.allclose(_tef[_te.isna()].values, _full_med)),
])

Кирпичик 2 — импутация без утечки
  [PASS] В train не осталось пропусков
  [PASS] В test не осталось пропусков
  [PASS] Пропуски заполнены медианой TRAIN
  [PASS] Медиана train != медианы всех данных (есть чем отличить утечку)
  [PASS] Не использована медиана всех данных (нет утечки)
  -> 5/5 проверок пройдено


### Кирпичик 3 — One-Hot кодирование категорий

**`OneHotEncoder`** — превращает категориальный столбец в набор бинарных столбцов-индикаторов
(по одному на категорию). Обучается на train (`fit_transform`), применяется к test (`transform`);
`handle_unknown="ignore"` безопасно обрабатывает не встречавшиеся категории.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

**Задание.** Реализуйте `one_hot(train_cat, test_cat)`: обучите энкодер на train и примените к train и test.
Верните `(enc_train, enc_test)` как numpy-массивы.

In [7]:
X_train[CAT_COLS[0]][:10].values.reshape(-1, 1)

array([['small'],
       ['medium'],
       ['large'],
       ['medium'],
       ['small'],
       ['small'],
       ['medium'],
       ['large'],
       ['medium'],
       ['small']], dtype=object)

In [8]:
OneHotEncoder().fit_transform(X_train[CAT_COLS[0]][:10].values.reshape(-1, 1)).toarray()

array([[0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [9]:
def one_hot(train_cat: pd.Series, test_cat: pd.Series):
    # TODO: обучите OneHotEncoder на train_cat, примените к train_cat и test_cat
    #       верните два numpy-массива (для train и для test)
    train_cat_arr = train_cat.values.reshape(-1, 1)
    test_cat_arr = test_cat.values.reshape(-1, 1)
    enc = OneHotEncoder(handle_unknown='ignore')
    enc.fit(train_cat_arr)
    return enc.transform(train_cat_arr).toarray(), enc.transform(test_cat_arr).toarray()

In [10]:
_a, _b = one_hot(X_train[CAT_COLS[0]], X_test[CAT_COLS[0]])
_ref = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
_ra = _ref.fit_transform(X_train[[CAT_COLS[0]]])
_rb = _ref.transform(X_test[[CAT_COLS[0]]])

run_checks("Кирпичик 3 — One-Hot кодирование", [
    ("Число колонок = число категорий train", lambda: np.asarray(_a).shape[1] == len(_ref.categories_[0])),
    ("train закодирован верно",               lambda: np.allclose(np.asarray(_a), _ra)),
    ("test закодирован тем же энкодером",      lambda: np.allclose(np.asarray(_b), _rb)),
    ("В каждой строке train ровно один бит",   lambda: np.allclose(np.asarray(_a).sum(axis=1), 1)),
])

Кирпичик 3 — One-Hot кодирование
  [PASS] Число колонок = число категорий train
  [PASS] train закодирован верно
  [PASS] test закодирован тем же энкодером
  [PASS] В каждой строке train ровно один бит
  -> 4/4 проверок пройдено


### Кирпичик 4 — метрики из ошибок (руками)

По матрице ошибок (`TP/FP/FN/TN`) считаются:
$\text{precision}=\frac{TP}{TP+FP}$, $\text{recall}=\frac{TP}{TP+FN}$,
$F_1$ — гармоническое среднее precision и recall, $\text{accuracy}=\frac{TP+TN}{\text{всего}}$.
Эталонные реализации:
[precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) ·
[recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) ·
[f1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) ·
[accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) ·
[confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

**Задание.** Реализуйте `manual_metrics(y_true, y_pred)` (класс 1 — положительный) и верните словарь
с ключами `"precision"`, `"recall"`, `"f1"`, `"accuracy"`.

In [ ]:
def manual_metrics(y_true, y_pred):
    # TODO: посчитайте TP, FP, FN, TN и по ним precision, recall, f1, accuracy
    TP = 0
    TN = 0
    FN = 0
    FP = 0
    for i in range(len(y_pred)):

        FN += y_true[i] ^ y_pred[i] and (not y_pred[i])
        FP += y_true[i] ^ y_pred[i] and y_pred[i]
        TP +=  not(y_true[i] ^ y_pred[i]) and y_pred[i]
        TN +=  not(y_true[i] ^ y_pred[i]) and (not y_pred[i])

        # if y_true[i] != y_pred[i]:
        #     if y_true[i]:
        #         FN += 1
        #     else:
        #         FP += 1
        # else:
        #     if y_true[i]:
        #         TP += 1
        #     else:
        #         TN += 1
        
        

    return {
        "precision": TP / (TP + FP),
        "recall": TP / (TP + FN),
        "f1": 2 * TP  / (2 * TP + FP + FN),
        "accuracy": (TP + TN) / len(y_true)
    }

In [12]:
_yt = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
_yp = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])
_m = manual_metrics(_yt, _yp)

run_checks("Кирпичик 4 — метрики вручную", [
    ("precision верна", lambda: abs(_m["precision"] - precision_score(_yt, _yp)) < 1e-9),
    ("recall верна",    lambda: abs(_m["recall"]    - recall_score(_yt, _yp))    < 1e-9),
    ("f1 верна",        lambda: abs(_m["f1"]        - f1_score(_yt, _yp))        < 1e-9),
    ("accuracy верна",  lambda: abs(_m["accuracy"]  - accuracy_score(_yt, _yp))  < 1e-9),
])

Кирпичик 4 — метрики вручную
  [PASS] precision верна
  [PASS] recall верна
  [PASS] f1 верна
  [PASS] accuracy верна
  -> 4/4 проверок пройдено


### Кирпичик 5 — почему не accuracy при дисбалансе

**`roc_auc_score(y_true, proba)`** — площадь под ROC-кривой. Интерпретация: вероятность, что
случайный положительный объект получит больший скор, чем случайный отрицательный. Принимает
вероятности/скор, а не метки. AUC = 0.5 — случайная модель.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

**Задание.** Реализуйте `auc_and_dummy(y_true, proba)`: верните `(roc_auc, dummy_accuracy)`, где
`dummy_accuracy` — доля мажоритарного класса (accuracy тривиального «всегда мажоритарный класс»).

In [18]:
a = list(zip([1, 0, 1], [0.3, 0.7, 0.5]))
a.sort(key=lambda x: x[1])
print(a)


[(1, 0.3), (1, 0.5), (0, 0.7)]


In [120]:
def auc_and_dummy(y_true, proba):
    # TODO: roc_auc по proba; dummy_accuracy = доля мажоритарного класса
    
    n = len(y_true)

    for i in range(n):
        if not y_true[i]:
            proba[i] -= 1e-9 

    prob_sorted = np.argsort(proba)

   

    greater = 0

    n_false = 0

    n_true = np.count_nonzero(y_true).item()

    for i in prob_sorted:

        if y_true[i]: 
            greater += n_false

        if not y_true[i]:
            n_false += 1

    n_eq_proba = 1
    prev_proba = -1

    eq = 0 

    for i in prob_sorted:
        if not y_true[i]:
            if proba[i] - prev_proba < 1e-8:
                n_eq_proba += 1
            prev_proba = proba[i]
        else:
            if proba[i] - prev_proba < 1e-8:
                eq += n_eq_proba
            prev_proba = -1
            n_eq_proba = 1

    denom =  n_true * (n - n_true)

    auc = (greater - 0.5 * eq) / denom
    return auc, max(n_false / n, (n - n_false) / n)

In [121]:
b = [0, 0, 0, 1, 1, 1, 0]
c = [.5, .1, .2, .6, .2, .3, 0]
auc_and_dummy(b, c)

(0.7916666666666666, 0.5714285714285714)

In [49]:
np.argsort(c)

array([5, 4, 3, 2, 1, 0])

In [115]:
roc_auc_score(b, c)

0.8333333333333334

In [105]:
_yt = np.array([0] * 90 + [1] * 10)
_proba = np.concatenate([np.linspace(0.10, 0.60, 90), np.linspace(0.40, 0.95, 10)])
_auc, _dummy = auc_and_dummy(_yt, _proba)

run_checks("Кирпичик 5 — ROC-AUC vs accuracy при дисбалансе", [
    ("ROC-AUC верна",                          lambda: abs(_auc - roc_auc_score(_yt, _proba)) < 1e-9),
    ("Dummy-accuracy = доля мажор. класса",    lambda: abs(_dummy - 0.90) < 1e-9),
    ("Урок: dummy-accuracy >= 0.9, но модель бесполезна", lambda: _dummy >= 0.9),
])

Кирпичик 5 — ROC-AUC vs accuracy при дисбалансе
  [PASS] ROC-AUC верна
  [PASS] Dummy-accuracy = доля мажор. класса
  [PASS] Урок: dummy-accuracy >= 0.9, но модель бесполезна
  -> 3/3 проверок пройдено


### Кирпичик 6 — честная кросс-валидация

**`cross_val_score(estimator, X, y, cv, scoring)`** — обучает и оценивает модель на нескольких
фолдах, возвращает массив оценок (по одной на фолд).
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
**`StratifiedKFold`** — разбиение на фолды с сохранением долей классов (объект `CV` уже готов).
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)

**Задание.** Реализуйте `cv_auc(estimator, X, y)`: оцените `estimator` по метрике `roc_auc`
на 5 фолдах (`cv=CV`). Верните массив из 5 значений.

In [132]:
def cv_auc(estimator, X, y):
    # TODO: оцените estimator стратифицированной 5-fold кросс-валидацией (объект CV)
    #       по метрике roc_auc; верните массив из 5 значений
    return cross_val_score(estimator, X, y, cv=CV, scoring="roc_auc")

In [133]:
_scores = cv_auc(make_estimator(), Xn, y)
_ref = cross_val_score(make_estimator(), Xn, y, cv=CV, scoring="roc_auc")

run_checks("Кирпичик 6 — стратифицированная кросс-валидация", [
    ("5 значений (5 фолдов)",                    lambda: len(_scores) == 5),
    ("Совпадает с эталоном (тот же seed/метрика)", lambda: np.allclose(np.sort(_scores), np.sort(_ref))),
    ("Средний ROC-AUC разумный (> 0.9)",         lambda: float(np.mean(_scores)) > 0.9),
])

Кирпичик 6 — стратифицированная кросс-валидация
  [PASS] 5 значений (5 фолдов)
  [PASS] Совпадает с эталоном (тот же seed/метрика)
  [PASS] Средний ROC-AUC разумный (> 0.9)
  -> 3/3 проверок пройдено


### Кирпичик 7 — подбор гиперпараметра (GridSearchCV)

**`GridSearchCV(estimator, param_grid, cv, scoring)`** — перебирает все комбинации гиперпараметров
из сетки с кросс-валидацией; после `fit` лучшие лежат в `best_params_`, лучшая модель — в `best_estimator_`.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

**Задание.** Реализуйте `best_C(X, y)`: переберите сетку `PARAM_GRID` (базовый оценщик `make_estimator()`,
`cv=CV`, `scoring="roc_auc"`). Верните лучшее значение `C` из `best_params_["logisticregression__C"]`.

In [ ]:
np.logspace()

In [136]:
def best_C(X, y):
    # TODO: переберите PARAM_GRID через GridSearchCV (оценщик make_estimator(), cv=CV,
    #       метрика roc_auc); верните лучшее C из best_params_
    return GridSearchCV(estimator=make_estimator(), cv=CV, param_grid=PARAM_GRID, scoring="roc_auc").fit(X, y)
    # return None

In [137]:
_grid = best_C(Xn, y)

In [141]:
_grid

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=5000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'logisticregression__C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbos

In [139]:
_grid.best_params_["logisticregression__C"]

10

In [140]:
_bc = 10
_ref_c = GridSearchCV(make_estimator(), PARAM_GRID, cv=CV, scoring="roc_auc").fit(Xn, y).best_params_["logisticregression__C"]

run_checks("Кирпичик 7 — GridSearchCV подбор C", [
    ("C из заданной сетки",   lambda: _bc in C_GRID),
    ("C совпал с эталоном",   lambda: _bc == _ref_c),
])

Кирпичик 7 — GridSearchCV подбор C
  [PASS] C из заданной сетки
  [PASS] C совпал с эталоном
  -> 2/2 проверок пройдено


### Кирпичик 8 — важность признаков (топ-3)

**`DecisionTreeClassifier`** — решающее дерево. После `fit` атрибут `feature_importances_`
показывает вклад каждого признака (сумма = 1); чем больше, тем важнее признак для модели.
[Документация →](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

**Задание.** Реализуйте `top3_features(X, y)`: обучите `DecisionTreeClassifier(random_state=SEED)`
и верните список из 3 названий признаков с наибольшей важностью (по убыванию).

In [ ]:
def top3_features(X, y):
    # TODO: обучите DecisionTreeClassifier(random_state=SEED); возьмите feature_importances_
    #       верните 3 названия столбцов с наибольшей важностью (по убыванию)
    
    return None

In [142]:
_t = DecisionTreeClassifier(random_state=SEED).fit(Xn, y)

In [145]:
[1, 2, 3, 4][-2:]

[3, 4]

In [146]:
Xn.columns[np.argsort(_t.feature_importances_)[-3:]]

Index(['mean perimeter', 'mean texture', 'mean area'], dtype='str')

In [147]:
_ref_top = list(np.array(Xn.columns)[np.argsort(_t.feature_importances_)[::-1]][:3])

In [148]:
_ref_top

['mean area', 'mean texture', 'mean perimeter']

In [ ]:
_top = top3_features(Xn, y)
_t = DecisionTreeClassifier(random_state=SEED).fit(Xn, y)
_ref_top = list(np.array(Xn.columns)[np.argsort(_t.feature_importances_)[::-1]][:3])

run_checks("Кирпичик 8 — важность признаков (топ-3)", [
    ("Вернулось 3 признака",         lambda: len(_top) == 3),
    ("Это реальные столбцы",         lambda: all(f in list(Xn.columns) for f in _top)),
    ("Топ-3 совпал с эталоном",      lambda: list(_top) == _ref_top),
])